# graph_builder.ipynb
# Loads pre-built features from features.ipynb and constructs the rolling
# correlation graphs (edge_index + edge_weight) for each timestep.
#
# Upstream outputs consumed:
#   data/processed/features.npy          (T, N, F) normalized feature tensor
#   data/processed/targets_ranked.npy    (T, N)    cross-sectional ranked target
#   data/processed/dates.csv             (T,)      aligned date index
#   data/processed/tickers.csv           (N,)      ticker list
#   data/log_returns.csv                           raw log-returns (for corr)


In [4]:
import pandas as pd
import numpy as np
import os

# ── Load pre-built features & metadata from features.ipynb ───────────────────
BASE = "data/processed"

features_norm   = np.load(f"{BASE}/features.npy")          # (T, N, F)
targets_ranked  = np.load(f"{BASE}/targets_ranked.npy")    # (T, N)
dates          = pd.to_datetime(pd.read_csv(f"{BASE}/dates.csv").iloc[:, 0])
tickers        = pd.read_csv(f"{BASE}/tickers.csv").iloc[:, 0].tolist()
feature_names  = pd.read_csv(f"{BASE}/feature_names.csv").iloc[:, 0].tolist()

n_timesteps, n_stocks, n_features = features_norm.shape

print(f"features_norm  : {features_norm.shape}   (T, N, F)")
print(f"targets_ranked : {targets_ranked.shape}   (T, N)")
print(f"dates          : {len(dates)}  ({dates.iloc[0].date()} → {dates.iloc[-1].date()})")
print(f"tickers        : {len(tickers)}")
print(f"feature_names  : {feature_names}")

# ── Load raw log-returns for correlation computation ─────────────────────────
log_returns = pd.read_csv("data/log_returns.csv", index_col=0, parse_dates=True)
log_returns = log_returns[tickers]   # align column order to feature tensor

print(f"\nlog_returns    : {log_returns.shape}")


features_norm  : (2003, 462, 15)   (T, N, F)
targets_ranked : (2003, 462)   (T, N)
dates          : 2003  (2015-12-30 → 2025-12-01)
tickers        : 462
feature_names  : ['roll_mean_5', 'roll_mean_20', 'roll_std_5', 'roll_std_20', 'mom_12_1', 'mom_6_1', 'mom_3_1', 'mom_1_0', 'reversal_1w', 'sharpe_20', 'sharpe_60', 'rel_strength_20', 'rel_strength_60', 'vol_weighted_mom_5', 'vol_weighted_mom_20']

log_returns    : (2764, 462)


In [5]:
# ── Build rolling correlation graphs ─────────────────────────────────────────
# For each timestep t, compute the 60-day rolling Pearson correlation matrix
# over the log_returns window ending at dates[t].
# Edges are created for all stock pairs with correlation > threshold.

LOOKBACK  = 60
THRESHOLD = 0.5

edge_indices = []   # list of (2, E) int64 arrays
edge_weights = []   # list of (E,)   float32 arrays

for t in range(n_timesteps):
    date = dates.iloc[t]
    pos  = log_returns.index.get_loc(date)

    start  = max(0, pos - LOOKBACK)
    window = log_returns.iloc[start:pos].values   # (<=60, N)

    if window.shape[0] < 2:
        edge_indices.append(np.zeros((2, 0), dtype=np.int64))
        edge_weights.append(np.zeros(0, dtype=np.float32))
        continue

    corr = np.corrcoef(window.T)                  # (N, N)
    corr = np.nan_to_num(corr, nan=0.0)

    rows, cols = np.where((corr > THRESHOLD) & (np.eye(n_stocks) == 0))
    edge_indices.append(np.stack([rows, cols], axis=0).astype(np.int64))
    edge_weights.append(corr[rows, cols].astype(np.float32))

    if t % 250 == 0:
        print(f"t={t:4d}/{n_timesteps}  edges: {edge_indices[-1].shape[1]:6d}")

print("\nDone building graphs")


t=   0/2003  edges:  34940


C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


t= 250/2003  edges:  10134
t= 500/2003  edges:  46850
t= 750/2003  edges:  28850
t=1000/2003  edges:  20568
t=1250/2003  edges: 126008
t=1500/2003  edges:  21814
t=1750/2003  edges:  10696
t=2000/2003  edges:   7514

Done building graphs


In [10]:
# ── Train / val / test split ──────────────────────────────────────────────────
# Splits are defined on the dates index from features.ipynb.

TRAIN_END = pd.Timestamp("2023-01-01")
VAL_END   = pd.Timestamp("2024-01-01")

train_mask = dates < TRAIN_END
val_mask   = (dates >= TRAIN_END) & (dates < VAL_END)
test_mask  = dates >= VAL_END

train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

print(f"Train: {len(train_idx):4d} timesteps  "
      f"({dates.iloc[train_idx[0]].date()} → {dates.iloc[train_idx[-1]].date()})")
print(f"Val:   {len(val_idx):4d} timesteps  "
      f"({dates.iloc[val_idx[0]].date()}  → {dates.iloc[val_idx[-1]].date()})")
print(f"Test:  {len(test_idx):4d} timesteps  "
      f"({dates.iloc[test_idx[0]].date()}  → {dates.iloc[test_idx[-1]].date()})")


Train: 1295 timesteps  (2015-12-30 → 2022-12-30)
Val:    240 timesteps  (2023-01-03  → 2023-12-29)
Test:   468 timesteps  (2024-01-02  → 2025-12-01)


In [11]:
# ── Save all processed data ───────────────────────────────────────────────────
os.makedirs("data/processed", exist_ok=True)

# features & targets are already in data/processed from features.ipynb;
# we only need to save the graph arrays and split indices.

edge_indices_arr = np.empty(len(edge_indices), dtype=object)
edge_weights_arr = np.empty(len(edge_weights), dtype=object)
for i in range(len(edge_indices)):
    edge_indices_arr[i] = edge_indices[i]
    edge_weights_arr[i] = edge_weights[i]

np.save("data/processed/edge_indices.npy", edge_indices_arr)
np.save("data/processed/edge_weights.npy", edge_weights_arr)
np.save("data/processed/train_idx.npy",    train_idx)
np.save("data/processed/val_idx.npy",      val_idx)
np.save("data/processed/test_idx.npy",     test_idx)

print("Saved:")
print(f"  edge_indices.npy  {len(edge_indices_arr)} graphs")
print(f"  edge_weights.npy  {len(edge_weights_arr)} graphs")
print(f"  train_idx.npy     {train_idx.shape}")
print(f"  val_idx.npy       {val_idx.shape}")
print(f"  test_idx.npy      {test_idx.shape}")


Saved:
  edge_indices.npy  2003 graphs
  edge_weights.npy  2003 graphs
  train_idx.npy     (1295,)
  val_idx.npy       (240,)
  test_idx.npy      (468,)


In [12]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"features_norm  : {features_norm.shape}  (T, N, F)")
print(f"targets_ranked : {targets_ranked.shape}  (T, N)")
print(f"NaNs in features : {np.isnan(features_norm).sum()}")
print(f"NaNs in targets  : {np.isnan(targets_ranked).sum()}")

t_s = train_idx[100]
print(f"\nSample graph at t={t_s}  ({dates.iloc[t_s].date()}):")
print(f"  edges:           {edge_indices[t_s].shape[1]}")
print(f"  avg edge weight: {edge_weights[t_s].mean():.3f}")

flat = targets_ranked[train_idx].flatten()
print(f"\nTarget stats (train):")
print(f"  mean={flat.mean():.4f}  std={flat.std():.4f}  "
      f"min={flat.min():.4f}  max={flat.max():.4f}")


features_norm  : (2003, 462, 15)  (T, N, F)
targets_ranked : (2003, 462)  (T, N)
NaNs in features : 0
NaNs in targets  : 0

Sample graph at t=100  (2017-05-30):
  edges:           10252
  avg edge weight: 0.592

Target stats (train):
  mean=0.0000  std=0.5786  min=-1.0000  max=1.0000
